# Laplacien, diffusion et réseau de neurones sur graphe

Ce notebook accompagne les exercices 10.1, 10.2 et 10.3. Le Laplacien est le fil conducteur : il mesure les variations d'un signal, engendre une diffusion, se transforme naturellement lorsque les sommets sont renumérotés et fournit l'opérateur de lissage d'une couche de convolution sur graphe.

## Parcours

1. [Laplacien et énergie](#laplacien)
2. [Spectre et diffusion](#spectre)
3. [Permutation des sommets](#permutation)
4. [Une première classification de sommets](#gcn)

In [1]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

<a id="laplacien"></a>
## 1. Laplacien et énergie

On considère le graphe chemin $1-2-3-4$, avec des poids égaux à un. Pour un signal $u$, vérifiez

$$
u^{\mathsf T}Lu
=
\frac12\sum_{i,j=1}^4 A_{ij}(u_i-u_j)^2.
$$

In [2]:
n = 4
A = jnp.zeros((n, n))
A = A.at[jnp.arange(n - 1), jnp.arange(1, n)].set(1.0)
A = A + A.T
u = jnp.array([1.0, -1.0, 2.0, 0.5])

print("A =")
print(A)
print("u =", u)

A =
[[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]
u = [ 1.  -1.   2.   0.5]


Construisez $D$ et $L=D-A$. Calculez l'énergie sous forme matricielle puis comme somme sur les arêtes. Vérifiez également que $L\mathbf 1=0$.

In [ ]:
# À compléter.

<a id="spectre"></a>
## 2. Spectre et diffusion

Diagonalisez $L$. Représentez ses vecteurs propres comme des signaux sur les sommets, puis calculez

$$
u(t)=e^{-tL}u(0).
$$

Ajoutez ensuite une arête de poids $0.2$ entre les sommets extrêmes et observez la variation de $\lambda_2$.

In [ ]:
# À compléter.

<a id="permutation"></a>
## 3. Permutation des sommets

La matrice $Q$ ci-dessous change la numérotation. Construisez

$$
A'=QAQ^{\mathsf T},\qquad
u'=Qu,
$$

recalculez $L'$ à partir de $A'$ et vérifiez

$$
L'=QLQ^{\mathsf T},
\qquad
{u'}^{\mathsf T}L'u'=u^{\mathsf T}Lu.
$$

In [5]:
permutation = jnp.array([2, 0, 3, 1])
Q = jnp.eye(n)[permutation]
print("Q =")
print(Q)

Q =
[[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]


In [ ]:
# À compléter.

<a id="gcn"></a>
## 4. Une première classification de sommets

Le graphe suivant comporte deux communautés de huit sommets. Les deux classes correspondent aux communautés. La première variable portée par un sommet est informative, mais plusieurs valeurs ont été perturbées au point de se trouver du mauvais côté de zéro.

Nous comparerons un classificateur affine appliqué directement à $H$ au même classificateur appliqué après le lissage

$$
\widehat A H,
\qquad
\widehat A=\widetilde D^{-1/2}\widetilde A\widetilde D^{-1/2}.
$$

In [7]:
n_communaute = 8
n = 2 * n_communaute
A = np.zeros((n, n))
for debut in (0, n_communaute):
    indices = np.arange(debut, debut + n_communaute)
    A[np.ix_(indices, indices)] = 1.0
    A[indices, indices] = 0.0
A[7, 8] = A[8, 7] = 0.15
A = jnp.asarray(A)

z = jnp.concatenate(
    [jnp.zeros(n_communaute, dtype=jnp.int32),
     jnp.ones(n_communaute, dtype=jnp.int32)]
)
signal = jnp.array(
    [-1.4, -0.8, 0.6, -1.1, 0.4, -0.9, -1.2, 0.5,
      1.2,  0.7, -0.5, 1.1, -0.3, 0.8, 1.3, -0.4]
)
H = jnp.column_stack([signal, jnp.ones(n)])
indices_apprentissage = jnp.array([0, 1, 8, 9])

print("sommets étiquetés :", indices_apprentissage)
print("étiquettes :", z[indices_apprentissage])

sommets étiquetés : [0 1 8 9]
étiquettes : [0 0 1 1]


Construisez $L$, puis $\widetilde A$, $\widetilde D$, $\widetilde L_{\rm sym}$ et $\widehat A$. Vérifiez $\widehat A=I-\widetilde L_{\rm sym}$ et représentez la première variable avant et après lissage.

In [ ]:
# À compléter.

Apprenez maintenant les paramètres $W,b$ du score affine $XW+b$ à partir des quatre sommets étiquetés. Faites-le une fois avec $X=H$, puis avec $X=\widehat A H$. Comparez les prédictions sur l'ensemble des sommets.

In [ ]:
# À compléter.

### Questions complémentaires

1. Pourquoi le lissage est-il utile ici ?
2. Que se passe-t-il si le poids de l'arête entre les communautés augmente ?
3. Appliquez plusieurs fois $\widehat A$ et observez le phénomène de sur-lissage (*'over-smoothing'*).
4. Permutez les sommets et vérifiez que les prédictions sont permutées de la même manière.

## Bilan

Le Laplacien traduit la géométrie du graphe en un opérateur positif. Son énergie mesure les variations le long des arêtes, son spectre ordonne les modes selon leur régularité et son exponentielle engendre une diffusion. Une couche de convolution utilise la même géométrie pour propager les variables avant d'apprendre une transformation. L'équivariance garantit que le résultat ne dépend pas de la numérotation des sommets.